# LAB: Practice with Tavily + LangChain Agents

### Objective
Reinforce your understanding of LangChain agents integrated with Tavily for real-time web search by solving two open-ended tasks. You'll:
- Load and configure a LangChain agent with Tavily
- Build effective prompts
- Generate structured, useful outputs from real-time data

### Setup

In [5]:
from langchain_classic import hub
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_core.tools import Tool
import os
import tiktoken
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_openai import ChatOpenAI

In [18]:
import os
from langchain_classic.agents import AgentExecutor, Tool
from langchain_classic.chat_models import ChatOpenAI
from langchain_classic.tools.tavily_search import TavilySearchResults
from langchain_core.prompts import ChatPromptTemplate , PromptTemplate


from IPython.display import Markdown, display
from  dotenv import load_dotenv

import warnings
warnings.filterwarnings('ignore')


load_dotenv()

os.environ["IRONHACKLABTAVILY_KEY"] 

tavily_search = TavilySearchResults()

# LLM + Encoding
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
encoding = tiktoken.encoding_for_model("gpt-4o-mini")

# Safe Tavily wrapper
def safe_search(query: str) -> str:
    result = tavily_search.run(query)

    # Ensure result is a string — Tavily returns dict with 'snippets' sometimes
    if isinstance(result, dict):
        result_text = result.get("content", "") or str(result)
    else:
        result_text = str(result)

    tokens = encoding.encode(result_text)
    trimmed = encoding.decode(tokens[:3500])  # leave room for GPT-4 response
    return trimmed


# LangChain tool
tools = [
    Tool(
        name="TavilySafeSearch",
        func=safe_search, 
        description="Web search tool"
    )
]

prompt = hub.pull("hwchase17/react")

""" prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a ReAct-style reasoning agent specialized in AI in Healthcare. "
            "You can use tools to look up information.\n\n"
            "You MUST follow this exact format whenever you think about using tools:\n\n"
            "Thought: you should always think about the question and how to answer \n"
            "Action: you will always use Tavily Search tool names given [{tool_names}]\n"
            "Action Input: Use the input and break it down in to simpler sentences \n"
            "Observation: Organize the result of action input as defined in prompt\n"
            #"... repeat Thought/Action/Action Input/Observation as needed\n"
            "When you are done using tools, finish with:\n"
            "Final Answer: the final answer to the user's question\n\n"
            "Available tools:\n{tools}\n"
        ),
        ("human", "Question: {input}"),
        ("ai", "{agent_scratchpad}"),
    ]
) """


agent = create_react_agent(llm=llm,tools=tools,prompt=prompt)

agent_executor = AgentExecutor(
    agent = agent,
    tools = tools,
    verbose = True,
    handle_parsing_errors = True,
    #max_iterations=10     
)


#agent = create_react_agent(tools=tools, llm=llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True)


In [ ]:
""" # define system prompt
system_prompt = "Use tavily API or Agent for most recent information retrieval or time based"
# Get the React prompt from hub
base_prompt = hub.pull("hwchase17/react")
new_template = system_prompt + "\n\n" + base_prompt.template
prompt = PromptTemplate(
    input_variables=base_prompt.input_variables,
    template=new_template,
) """

### Exercise 1: AI in Healthcare

Goal: Investigate and summarize the latest advancements in generative AI applied to healthcare in 2025.

- Design a prompt that asks the agent to retrieve the most recent updates.
- Ensure the agent outputs a structured response in Markdown.

In [ ]:
# Your prompt here
prompt_1 = "Todays date is : 02.03.2026 . Research, Investigate and Summarize applications in the field of Generative AI in HEALTHCARE. Then look for advancements in last 2 years(2024-2025) for the same field"
#prompt_1 = "Whats the date today?
#prompt_1 = "why there was strike 01.03.2026 in germany? and give me source as well"

# Run it
response_1 = agent_executor.invoke({"input":prompt_1})





> Entering new AgentExecutor chain...
I need to gather information on the applications of Generative AI in healthcare, focusing on advancements made in the years 2024 and 2025. This will require a web search to find relevant and up-to-date information on the topic.

Action: TavilySafeSearch
Action Input: "Generative AI applications in healthcare advancements 2024 2025"
[{'title': 'Generative Artificial Intelligence Use in Healthcare - PMC - NIH', 'url': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC11739231/', 'content': 'reveals that approximately 75% of large healthcare organizations are currently using or planning to scale up generative AI in their operations . In 2023, Gen AI technologies made substantial advancements, as evidenced by a recent study in which ChatGPT surpassed physicians in both quality and empathy metrics , and Google’s MedPaLM-2 LLM achieved expert-level scores on the United States Medical Licensing Examination . Furthermore, the first drug entirely designed with Ge

In [23]:
display(Markdown(response_1['output']))


Generative AI is being applied in various healthcare domains, including clinical documentation, patient communication, diagnosis support, drug discovery, and medical education. Significant advancements from 2024 to 2025 include the entry of AI-designed drugs into clinical trials, improved diagnostic tools, and the mainstream adoption of AI decision-making tools. Ethical considerations and collaboration between tech and healthcare sectors are crucial for the responsible implementation of these technologies.

### Exercise 2: AI Startups Landscape

Goal: Track 2025’s top emerging AI startups and their innovations.

- Create a prompt that instructs the agent to deliver a clean Markdown summary.
- Tip: Ask for company names, product highlights, and sources.

In [24]:
# Your prompt here
prompt_2 = "Todays date is : 02.03.2026 . Research, Investigate and Summarize applications in the field of AI startup Landscape. Then look for company names, product highlights and sources"

# Run it

response_2 = agent_executor.invoke({"input":prompt_2})



> Entering new AgentExecutor chain...
I need to gather information about the current landscape of AI startups, including their applications, notable companies, and product highlights. Since my knowledge is limited to data up to October 2023, I will perform a web search to find the most recent information on AI startups as of March 2026.

Action: TavilySafeSearch
Action Input: "AI startup landscape applications March 2026"
[{'title': 'The AI Landscape: March 2026 - by Jordamøn - AI Central', 'url': 'https://aicentral.substack.com/p/the-ai-landscape-march-2026', 'content': 'curiosity. OpenAI’s competing coding product, Codex, has passed 1.5 million weekly active users. As part of its Amazon deal, OpenAI is developing a “stateful runtime environment” on Bedrock with an additional $100 billion in compute commitments, binding the two companies’ infrastructure fortunes together at a scale that would be difficult to unwind. [...] Meta’s trajectory is harder to read. After building its AI st

In [26]:
display(Markdown(response_2['output']))
#display(Markdown(response_1['output']))

As of March 2026, the AI startup landscape is characterized by significant growth and innovation, with notable companies like OpenAI, Anthropic, BharatGen, and MiniMax leading the way. Key applications include chatbots, healthcare diagnostics, and multilingual AI models, while challenges such as data acquisition and competition persist. The industry is poised for further expansion, driven by the urgent need for automation and efficiency in various sectors.

### Exercice 3: Compare Two Tech Products
- **Prompt idea:** Compare the key features, pricing, and reviews of OpenAI’s ChatGPT Team and Anthropic’s Claude Pro.
- Ensure the agent outputs a structured response in Markdown.




In [27]:
# Your prompt here
prompt_3 = "Compare the key features, pricing, and reviews of OpenAI’s ChatGPT Team and Anthropic’s Claude Pro "

# Run it
response_3 = agent_executor.invoke({"input":prompt_3})




> Entering new AgentExecutor chain...
To compare the key features, pricing, and reviews of OpenAI’s ChatGPT Team and Anthropic’s Claude Pro, I will need to gather information about both products. This will involve searching for their features, pricing details, and user reviews. 

Action: TavilySafeSearch  
Action Input: "OpenAI ChatGPT Team features pricing reviews"  [{'title': 'ChatGPT Plans Compared: Free vs Plus ($20) vs Pro ($200) vs ...', 'url': 'https://intuitionlabs.ai/articles/chatgpt-plans-comparison', 'content': 'In January 2024, OpenAI filled the “middle” market with ChatGPT Teams (later renamed Business), a self-serve plan for small and mid-size teams (( "Highlights: Product")) (( "Highlights: How does the pricing work,for ChatGPT Business")). Teams offered collaboration (shared workspace, custom GPTs, admin tools) along with advanced models (GPT-4 with 32K context, premium DALL·E-3 images, browsing, data analysis) (( "Highlights: ChatGPT Team includes:")) (( "Highlights:

In [28]:
#display(Markdown(response_3))
display(Markdown(response_3['output']))

OpenAI's ChatGPT Team (Business) costs $25/user/month (annual) with features like collaboration tools and advanced models, while Anthropic's Claude Pro costs $17/month (annual) and offers 5x usage, memory, and integration with tools like Excel. Both have positive reviews, with ChatGPT noted for integrations and Claude Pro for reasoning capabilities.

## Bonus Task: Propose and Implement Your Own Use Case

As a final challenge, think of a real-world scenario where an AI agent could provide value using web search or external tools.


In [30]:
# Your prompt here
prompt_4 = "think of a real-world scenario where an AI agent could provide value using web search or external tools "

# Run it
response_4 = agent_executor.invoke({"input":prompt_4})



> Entering new AgentExecutor chain...
I need to think of a specific real-world scenario where an AI agent can utilize web search or external tools to provide value. One potential scenario could be in the context of travel planning, where an AI agent helps users find the best travel options, accommodations, and activities based on their preferences and budget. 

Action: TavilySafeSearch
Action Input: "best travel planning tools 2023"
[{'title': '14 Best AI Travel Planning Tools Free & Paid [Updated 2026]', 'url': 'https://codeconductor.ai/blog/top-ai-travel-planning-tools/', 'content': '### 9. AskLAYLA\n\nOne of the standout tools in AI-powered travel planning is askLAYLA. This platform offers a comprehensive travel planning experience, from initial destination inspiration to final booking. What makes askLAYLA unique is its conversational AI interface, which learns your preferences—like budget, preferred activities, and travel style—over time. It also incorporates real-time feedback f

In [31]:
display(Markdown(response_4['output']))

An AI agent can provide value in travel planning by utilizing web search to recommend the best AI travel planning tools, such as askLAYLA, Tripnotes, and Roam Around, which help users find accommodations, activities, and manage their itineraries based on their preferences and budget.